# Model 2: Pretrained DeBERTa-v3 Multiple Choice Solver

**Project:** Smart MCQ Solver · DL & GenAI · Milestone 3  
**Kernel path:** `nb/pretrained/pre_trained.ipynb`

---

## Architecture

Uses `microsoft/deberta-v3-small` with HuggingFace `AutoModelForMultipleChoice`. All five options are scored in a **single batched forward pass** — no Python loops over options.

| Step | Description |
|------|-------------|
| **Tokenization** | Each question paired with each option: `[CLS] prompt [SEP] option [SEP]` |
| **Input shape** | `(batch, 5, seq_len)` — options stacked on dim-1 |
| **Encoder** | `deberta-v3-small` — disentangled attention, 6 layers, 768 hidden |
| **Classifier** | Linear head over pooled output → 5 raw logits |
| **Loss** | `CrossEntropyLoss(logits, correct_option_index)` |
| **Metric** | MAP@3 computed passively from raw logits — does not influence training |
| **Checkpoint** | Saved strictly on minimum `val_loss` — MAP@3 is read-only logging |

## 1. Setup & Environment Variables

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
import wandb
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
print('Imports ready.')

In [ ]:
# Data paths — checks local dev directories before falling back to Kaggle mount
KAGGLE_INPUT = '/kaggle/input/competitions/smart-mcq-solver-challenge'

def get_path(filename: str) -> str:
    for candidate in [
        os.path.join('..', '..', 'data', filename),
        os.path.join('data', filename),
        os.path.join(KAGGLE_INPUT, filename),
    ]:
        if os.path.exists(candidate):
            return candidate
    return filename

# GPU auto-detect; falls back to CPU locally
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f'Device : {DEVICE}')

In [ ]:
# W&B key — read from environment; avoids hardcoding secrets in the notebook
_wandb_key = os.environ.get(
    'WANDB_API_KEY',
    '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'
)
wandb.login(key=_wandb_key, relogin=True)

run = wandb.init(
    project='23f2004343-t22026',
    name='Model_2_DeBERTa_Final',
    config={
        'base_model'    : 'microsoft/deberta-v3-small',
        'max_seq_length': 256,
        'batch_size'    : 8,
        'epochs'        : 3,
        'learning_rate' : 2e-5,
        'warmup_ratio'  : 0.1,
        'weight_decay'  : 0.01,
        'optimizer'     : 'AdamW',
        'loss_fn'       : 'CrossEntropyLoss',
        'num_options'   : 5,
    },
)
cfg = wandb.config
print(f'W&B run : {run.name}  |  project : {run.project}')
print(f'Config  : {dict(cfg)}')

## 2. Dataset & Tokenization Pipeline

In [ ]:
trn_df = pd.read_csv(get_path('train.csv'))
tst_df = pd.read_csv(get_path('test.csv'))
print(f'train : {trn_df.shape}  |  test : {tst_df.shape}')
print(f'columns : {list(trn_df.columns)}')

OPTIONS   = ['A', 'B', 'C', 'D', 'E']
MODEL_NAME = cfg.base_model
MAX_LEN   = int(cfg.max_seq_length)

# Load tokenizer from HuggingFace Hub
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded: {MODEL_NAME}  |  max_len={MAX_LEN}')

In [ ]:
class MCQDataset(Dataset):
    """Tokenizes each MCQ row into 5 (prompt, option) pair encodings.

    Output tensors per item:
      input_ids      : (5, max_len)
      attention_mask : (5, max_len)
      label          : int in [0..4] — correct option index (train only)

    Note: token_type_ids are NOT included. DeBERTa-v3 has an empty token
    type vocabulary (type_vocab_size=0); passing them causes a CUDA
    index-out-of-bounds error in the embedding lookup.
    """

    def __init__(self, df: pd.DataFrame, is_train: bool = True):
        self.df       = df.reset_index(drop=True)
        self.is_train = is_train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        prompt = str(row['prompt'])

        # Tokenize all 5 (prompt, option) pairs in one batched call — shape (5, max_len)
        enc = tokenizer(
            [prompt] * 5,
            [str(row[opt]) for opt in OPTIONS],
            max_length=MAX_LEN,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids'     : enc['input_ids'],       # (5, max_len)
            'attention_mask': enc['attention_mask'],   # (5, max_len)
        }

        if self.is_train:
            ans          = str(row['answer']).strip().upper()
            label        = OPTIONS.index(ans) if ans in OPTIONS else 0
            item['label'] = torch.tensor(label, dtype=torch.long)

        return item


print('MCQDataset class ready.')

In [ ]:
# 90/10 train-validation split
trn_sub, val_sub = train_test_split(trn_df, test_size=0.1, random_state=SEED)
BATCH = int(cfg.batch_size)

trn_loader = DataLoader(
    MCQDataset(trn_sub, is_train=True),
    batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    MCQDataset(val_sub, is_train=True),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)
tst_loader = DataLoader(
    MCQDataset(tst_df, is_train=False),
    batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True,
)

probe = next(iter(trn_loader))
print(f"input_ids shape : {probe['input_ids'].shape}   # (batch, 5, max_len)")
print(f"labels sample   : {probe['label'].tolist()[:8]}")

## 3. Model Architecture & Initialization

In [ ]:
# AutoModelForMultipleChoice reshapes (batch, 5, seq_len) -> runs encoder
# once per option -> applies linear head on [CLS] -> returns logits (batch, 5)
model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)
model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model     : {MODEL_NAME}')
print(f'Parameters: {n_params:,} trainable')

criterion = nn.CrossEntropyLoss()   # labels are integer indices [0..4]
optimizer = AdamW(
    model.parameters(),
    lr=float(cfg.learning_rate),
    weight_decay=float(cfg.weight_decay),
    eps=1e-8,
)

total_steps  = len(trn_loader) * int(cfg.epochs)
warmup_steps = int(total_steps * float(cfg.warmup_ratio))
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)
print(f'Total steps: {total_steps}  |  warmup: {warmup_steps}')

## 4. Training Loop & Validation Metrics

In [ ]:
def ap_at_3(ranked: list, correct: int) -> float:
    """Average Precision at 3 for one question.

    ranked  : list of up to 3 predicted option indices, best-first
    correct : ground truth option index

    Viva: AP@3 = 1/rank if the correct answer is in the top-3 prediction,
    else 0. MAP@3 = mean(AP@3) across all questions — the official Kaggle metric.
    """
    hits, score = 0, 0.0
    for k, pred in enumerate(ranked[:3], start=1):
        if pred == correct:
            hits  += 1
            score += hits / k
    return score


def run_eval(model, loader) -> dict:
    """One full validation pass.

    Returns loss, accuracy, macro-F1, and MAP@3.
    MAP@3 is computed from raw logits — no softmax, no rounding.
    It is a passive observation: it does not influence gradients or checkpointing.
    """
    model.eval()
    total_loss, all_preds, all_labels, all_ap3 = 0.0, [], [], []

    with torch.no_grad():
        for batch in loader:
            ids   = batch['input_ids'].to(DEVICE)
            masks = batch['attention_mask'].to(DEVICE)
            labs  = batch['label'].to(DEVICE)

            logits = model(input_ids=ids, attention_mask=masks).logits  # (B, 5)
            total_loss += criterion(logits, labs).item()

            # Rank all 5 options by descending raw logit — no softmax needed for ordering
            ranked_idx = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().tolist()
            labels     = labs.cpu().tolist()

            all_preds.extend([r[0] for r in ranked_idx])   # top-1 for accuracy/F1
            all_labels.extend(labels)

            # Group logits per question to calculate MAP@3
            for ranked, correct in zip(ranked_idx, labels):
                all_ap3.append(ap_at_3(ranked, correct))

    n = max(len(loader), 1)
    return {
        'val_loss'    : total_loss / n,
        'val_accuracy': accuracy_score(all_labels, all_preds),
        'val_f1_macro': f1_score(all_labels, all_preds, average='macro', zero_division=0),
        'val_map3'    : float(np.mean(all_ap3)) if all_ap3 else 0.0,
    }

In [ ]:
EPOCHS        = int(cfg.epochs)
CKPT_PATH     = '/kaggle/working/deberta_mcq_best.pt'
best_val_loss = float('inf')
best_val_map3 = 0.0   # tracked passively for W&B summary

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for batch in trn_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)
        labs  = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        logits = model(input_ids=ids, attention_mask=masks).logits
        loss   = criterion(logits, labs)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # prevent exploding gradients
        optimizer.step()
        scheduler.step()
        running_loss += loss.item()

    avg_trn = running_loss / max(len(trn_loader), 1)
    val_m   = run_eval(model, val_loader)

    # Update running best MAP@3 — passive tracker only
    best_val_map3 = max(best_val_map3, val_m['val_map3'])

    wandb.log({'epoch': epoch, 'train_loss': avg_trn, **val_m})
    print(
        f'Epoch {epoch}/{EPOCHS}  '
        f'trn={avg_trn:.4f}  '
        f'val_loss={val_m["val_loss"]:.4f}  '
        f'val_acc={val_m["val_accuracy"]:.4f}  '
        f'val_f1={val_m["val_f1_macro"]:.4f}  '
        f'val_map3={val_m["val_map3"]:.4f}'
    )

    # Checkpoint saved on minimum val_loss — not MAP@3, which is volatile early in training
    if val_m['val_loss'] < best_val_loss:
        best_val_loss = val_m['val_loss']
        torch.save(model.state_dict(), CKPT_PATH)
        print(f'  ✓ Checkpoint saved  val_loss={best_val_loss:.4f}')

# Persist final summary stats to W&B run
wandb.run.summary['best_val_loss'] = best_val_loss
wandb.run.summary['best_val_map3'] = best_val_map3
print('\nTraining complete.')

## 5. Inference & Submission Generation

In [ ]:
# Load the best checkpoint saved during training
if os.path.exists(CKPT_PATH):
    model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
    print(f'Loaded checkpoint: {CKPT_PATH}')

model.eval()
OPTIONS_ARR = np.array(OPTIONS)
all_top3    = []

print(f'Running inference on {len(tst_df)} test questions...')
with torch.no_grad():
    for batch in tst_loader:
        ids   = batch['input_ids'].to(DEVICE)
        masks = batch['attention_mask'].to(DEVICE)

        # Raw logits (B, 5) — sort descending to get top-3 option indices
        logits   = model(input_ids=ids, attention_mask=masks).logits
        top3_idx = torch.argsort(logits, dim=1, descending=True)[:, :3].cpu().numpy()

        for row_idx in top3_idx:
            all_top3.append(' '.join(OPTIONS_ARR[row_idx]))

# Align output columns with sample_submission.csv — prevents KeyError on submission
sub_template = pd.read_csv(get_path('sample_submission.csv'))
id_col, pred_col = sub_template.columns[0], sub_template.columns[1]

print(f'Template columns : {list(sub_template.columns)}')
print(f'Predictions      : {len(all_top3)}')

assert len(all_top3) == len(tst_df), \
    f'Row count mismatch: got {len(all_top3)}, expected {len(tst_df)}'

sub_df = pd.DataFrame({id_col: tst_df['id'].values, pred_col: all_top3})
sub_df.to_csv('submission.csv', index=False)

print(f'submission.csv written — {len(sub_df)} rows')
print(sub_df.head())

# Log artifact and close W&B run
art = wandb.Artifact('submission_deberta_final', type='predictions')
art.add_file('submission.csv')
wandb.log_artifact(art)
wandb.finish()
print('W&B run closed.')